# Credit-risk modelling – Régression logistique (baseline)

**Objectif :** prédire la probabilité qu'un client fasse défaut le mois suivant, avec un modèle simple qui servira de **référence** (baseline) pour les autres modèles.

**Plan**
1. Chargement des données
2. Nettoyage
3. Feature engineering
4. Séparation train / test
5. Encodage et mise à l'échelle
6. Régression logistique : baseline puis correction du déséquilibre
7. Réglage de C par validation croisée
8. Choix du seuil de décision
9. Évaluation finale sur le test
10. Interprétation des coefficients
11. Variante sans variables sensibles

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, f1_score, precision_score, recall_score,
                             accuracy_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

# Dossiers du projet : fonctionne si le notebook est dans notebooks/ ou à côté des données
ROOT = Path.cwd().parent if (Path.cwd().parent / "data").exists() else Path.cwd()
FIG_DIR = ROOT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

ModuleNotFoundError: No module named 'numpy'

## 1. Chargement des données

Le fichier UCI a **deux lignes d'en-tête** : on lit les vrais noms de colonnes sur la 2ᵉ ligne (`header=1`).

In [ ]:
# Recherche du fichier de données (dans data/ ou à côté du notebook)
candidats = list((ROOT / "data").glob("default*credit*card*")) + list(Path.cwd().glob("default*credit*card*"))

DATA_PATH = candidats[0]
print("Fichier :", DATA_PATH)

if DATA_PATH.suffix == ".csv":
    df = pd.read_csv(DATA_PATH, header=1)
else:
    df = pd.read_excel(DATA_PATH, header=1)   # nécessite xlrd pour les .xls

print(df.shape)
df.head()

## 2. Nettoyage

- **Renommage** : `PAY_0` → `PAY_1` (même mois que `BILL_AMT1` et `PAY_AMT1`) et la cible → `DEFAULT`.
- **Suppression de `ID`** : un numéro de client ne dit rien sur le risque.
- **Doublons** : supprimés, pour qu'un même client ne soit pas à la fois dans le train et le test.
- **Codes non documentés** : `EDUCATION` 0, 5, 6 → 4 (autres) ; `MARRIAGE` 0 → 3 (autres).

In [ ]:
df = df.rename(columns={"PAY_0": "PAY_1", "default payment next month": "DEFAULT"})
df = df.drop(columns="ID")

print("Valeurs manquantes :", df.isna().sum().sum())
print("Doublons supprimés :", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

df["EDUCATION"] = df["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
df["MARRIAGE"] = df["MARRIAGE"].replace({0: 3})

print("Clients restants :", len(df))
print("Taux de défaut :", round(df["DEFAULT"].mean(), 3))

## 3. Feature engineering

On crée 4 variables métier qui résument le comportement du client sur les 6 mois :

| Variable | Calcul | Ce qu'elle mesure |
|------|------|------|
| `UTIL_MEAN` | moyenne de `BILL_AMT` / `LIMIT_BAL` | utilisation moyenne du crédit |
| `PAY_RATIO` | `PAY_AMT1` / `BILL_AMT2` (borné entre 0 et 1) | part de la facture d'août remboursée en septembre |
| `N_MONTHS_LATE` | nombre de mois avec `PAY_x` > 0 | fréquence des retards |
| `MAX_DELAY` | plus grand `PAY_x` | gravité du pire retard |

In [ ]:
bill_cols = [f"BILL_AMT{i}" for i in range(1, 7)]
payamt_cols = [f"PAY_AMT{i}" for i in range(1, 7)]
pay_cols = [f"PAY_{i}" for i in range(1, 7)]

df["UTIL_MEAN"] = df[bill_cols].mean(axis=1) / df["LIMIT_BAL"]

# Si la facture est nulle ou négative, il n'y a rien à rembourser : ratio = 1
df["PAY_RATIO"] = np.where(df["BILL_AMT2"] > 0, df["PAY_AMT1"] / df["BILL_AMT2"], 1)
df["PAY_RATIO"] = df["PAY_RATIO"].clip(0, 1)

df["N_MONTHS_LATE"] = (df[pay_cols] > 0).sum(axis=1)
df["MAX_DELAY"] = df[pay_cols].max(axis=1).clip(lower=0)

# Lien des nouvelles variables avec le défaut
df[["UTIL_MEAN", "PAY_RATIO", "N_MONTHS_LATE", "MAX_DELAY", "PAY_1", "DEFAULT"]].corr()["DEFAULT"].round(3)

## 4. Séparation train / test

80 % pour l'entraînement, 20 % pour le test. Le split est **stratifié** (`stratify=y`) : le train et le test gardent la même proportion de défauts (22 %).
Le test ne sera utilisé **qu'à la toute fin**.

In [ ]:
X = df.drop(columns="DEFAULT")
y = df["DEFAULT"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train :", X_train.shape, "| taux de défaut :", round(y_train.mean(), 3))
print("Test  :", X_test.shape, "| taux de défaut :", round(y_test.mean(), 3))

## 5. Encodage et mise à l'échelle

1. **One-hot encoding** de `SEX`, `EDUCATION`, `MARRIAGE` : ce sont des étiquettes, pas des grandeurs.
2. **Transformation log** de `LIMIT_BAL` et des `PAY_AMT` : ces montants sont très asymétriques (quelques valeurs énormes). Les `BILL_AMT` peuvent être négatifs, donc on ne leur applique pas le log.
3. **Standardisation** (moyenne 0, écart-type 1) : indispensable pour la régression logistique régularisée.

⚠️ Le `StandardScaler` est **ajusté sur le train uniquement**, puis appliqué au test, pour éviter toute fuite de données.

In [ ]:
def prepare(X):
    X = pd.get_dummies(X, columns=["SEX", "EDUCATION", "MARRIAGE"], drop_first=True, dtype=int)
    for col in ["LIMIT_BAL"] + payamt_cols:
        X[col] = np.log1p(X[col])
    return X

X_train_p = prepare(X_train)
X_test_p = prepare(X_test).reindex(columns=X_train_p.columns, fill_value=0)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_p)   # fit sur le train
X_test_s = scaler.transform(X_test_p)         # simple application sur le test

print("Nombre de variables après encodage :", X_train_s.shape[1])

Petite fonction pour calculer toutes les métriques d'un coup.

In [ ]:
def evaluate(name, y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {"Modèle": name, "Seuil": threshold,
            "ROC-AUC": roc_auc_score(y_true, proba),
            "F1": f1_score(y_true, pred),
            "Précision": precision_score(y_true, pred),
            "Rappel": recall_score(y_true, pred),
            "Accuracy": accuracy_score(y_true, pred)}

results = []

## 6. Régression logistique

### 6.1 Baseline brute

Régression logistique par défaut, sans correction du déséquilibre, seuil 0,5.

In [ ]:
lr_base = LogisticRegression(max_iter=1000)
lr_base.fit(X_train_s, y_train)
p_base = lr_base.predict_proba(X_test_s)[:, 1]

results.append(evaluate("LR baseline", y_test, p_base))
pd.DataFrame(results).round(3)

L'accuracy semble bonne, mais le **rappel est faible** : le modèle rate la majorité des défauts, car il privilégie la classe majoritaire.

### 6.2 Correction du déséquilibre : `class_weight="balanced"`

Avec 22 % de défauts, chaque erreur sur un défaut reçoit un poids environ **3,5 fois plus élevé** dans la fonction de perte. Le modèle est donc davantage pénalisé quand il rate un défaut.

In [ ]:
lr_bal = LogisticRegression(max_iter=1000, class_weight="balanced")
lr_bal.fit(X_train_s, y_train)
p_bal = lr_bal.predict_proba(X_test_s)[:, 1]

results.append(evaluate("LR balanced", y_test, p_bal))
pd.DataFrame(results).round(3)

## 7. Réglage de C par validation croisée

`C` contrôle la **régularisation** : petit C = modèle plus contraint, grand C = modèle plus libre.

On teste plusieurs valeurs avec une **validation croisée à 5 plis** sur le train : le train est coupé en 5 morceaux, on entraîne sur 4 et on valide sur le 5ᵉ, 5 fois de suite. On garde le C qui donne la meilleure AUC moyenne. Le test n'est pas utilisé.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid = GridSearchCV(
    LogisticRegression(max_iter=1000, class_weight="balanced"),
    param_grid={"C": [0.001, 0.01, 0.1, 1, 10, 100]},
    scoring="roc_auc",
    cv=cv,
)
grid.fit(X_train_s, y_train)

print("Meilleur C :", grid.best_params_["C"])
print("AUC moyenne en validation croisée :", round(grid.best_score_, 4))
pd.DataFrame({"C": grid.cv_results_["param_C"], "AUC moyenne (CV)": grid.cv_results_["mean_test_score"]}).round(4)

## 8. Choix du seuil de décision

Le seuil de 0,5 n'a rien d'obligatoire. On cherche celui qui **maximise le F1**.

Pour ne pas utiliser le test, on utilise `cross_val_predict` : chaque client du train reçoit une probabilité calculée par un modèle qui **ne l'a pas vu** pendant son entraînement.

In [ ]:
best_lr = grid.best_estimator_
p_train_cv = cross_val_predict(best_lr, X_train_s, y_train, cv=cv, method="predict_proba")[:, 1]

seuils = np.arange(0.10, 0.91, 0.01)
f1s = [f1_score(y_train, (p_train_cv >= s).astype(int)) for s in seuils]
best_seuil = round(float(seuils[np.argmax(f1s)]), 2)
print("Seuil retenu :", best_seuil)

plt.figure(figsize=(6, 3.5))
plt.plot(seuils, f1s)
plt.axvline(best_seuil, ls="--", c="grey")
plt.xlabel("Seuil"); plt.ylabel("F1 (validation croisée)")
plt.title("Choix du seuil de décision")
plt.tight_layout(); plt.savefig(FIG_DIR / "logreg_threshold.png", dpi=150); plt.show()

## 9. Évaluation finale sur le test

In [ ]:
p_best = best_lr.predict_proba(X_test_s)[:, 1]
results.append(evaluate("LR balanced + C optimisé + seuil optimisé", y_test, p_best, best_seuil))

res = pd.DataFrame(results).set_index("Modèle").round(3)
res

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

RocCurveDisplay.from_predictions(y_test, p_base, name="Baseline brute", ax=axes[0])
RocCurveDisplay.from_predictions(y_test, p_best, name="Modèle final", ax=axes[0])
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set_title("Courbe ROC (test)")

cm = confusion_matrix(y_test, (p_best >= best_seuil).astype(int))
ConfusionMatrixDisplay(cm, display_labels=["Pas de défaut", "Défaut"]).plot(ax=axes[1], cmap="Blues", colorbar=False)
axes[1].set_xlabel("Prédiction"); axes[1].set_ylabel("Réalité"); axes[1].grid(False)
axes[1].set_title(f"Matrice de confusion (seuil {best_seuil})")

plt.tight_layout(); plt.savefig(FIG_DIR / "logreg_evaluation.png", dpi=150); plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"Défauts détectés : {tp} sur {tp + fn} ({tp / (tp + fn):.0%})")
print(f"Défauts manqués : {fn} | Fausses alertes : {fp}")

## 10. Interprétation des coefficients

Les variables sont standardisées : quand une variable augmente d'un écart-type, la **cote du défaut est multipliée par $e^{\beta}$** (cours 4).
- coefficient positif (rouge) → augmente le risque ;
- coefficient négatif (bleu) → réduit le risque.

In [ ]:
coefs = pd.DataFrame({"variable": X_train_p.columns, "coef": best_lr.coef_[0]})
coefs["odds_ratio"] = np.exp(coefs["coef"])
coefs = coefs.reindex(coefs["coef"].abs().sort_values(ascending=False).index)

top = coefs.head(12).iloc[::-1]
plt.figure(figsize=(7, 4.5))
plt.barh(top["variable"], top["coef"], color=np.where(top["coef"] > 0, "#c0392b", "#2e86c1"))
plt.axvline(0, c="k", lw=0.8)
plt.xlabel("Coefficient")
plt.title("12 variables les plus influentes")
plt.tight_layout(); plt.savefig(FIG_DIR / "logreg_coefficients.png", dpi=150); plt.show()

coefs.head(12).round(3).reset_index(drop=True)

**Lecture :** les variables de **retard de paiement** (`N_MONTHS_LATE`, `MAX_DELAY`, `PAY_1`) augmentent le plus le risque. Un **plafond élevé** (`LIMIT_BAL`) et des **remboursements importants** (`PAY_AMT`) le réduisent.
`EDUCATION_4` peut ressortir fortement, mais c'est un **petit groupe** (moins de 500 clients) : il ne faut pas en tirer de conclusion.

## 11. Variante sans variables sensibles

`SEX`, `AGE` et `MARRIAGE` sont des caractéristiques protégées. On ré-entraîne le même modèle sans elles pour voir ce qu'on perd.

In [ ]:
sensibles = [c for c in X_train_p.columns if c.startswith(("SEX", "MARRIAGE")) or c == "AGE"]
X_train_f = X_train_p.drop(columns=sensibles)
X_test_f = X_test_p.drop(columns=sensibles)

scaler_f = StandardScaler()
lr_f = LogisticRegression(max_iter=1000, class_weight="balanced", C=grid.best_params_["C"])
lr_f.fit(scaler_f.fit_transform(X_train_f), y_train)
p_f = lr_f.predict_proba(scaler_f.transform(X_test_f))[:, 1]

results.append(evaluate("Sans SEX / AGE / MARRIAGE", y_test, p_f, best_seuil))
print(f"AUC avec : {roc_auc_score(y_test, p_best):.3f} | sans : {roc_auc_score(y_test, p_f):.3f}")

## Résultats finaux

Ces scores servent de **référence** : Random Forest, XGBoost et le MLP devront faire mieux.

In [ ]:
final = pd.DataFrame(results).set_index("Modèle").round(3)
final.to_csv(ROOT / "results" / "metrics_logreg.csv")
final